In [ ]:
import numpy as np
import os
os.environ["CUDA_DEVICE_ORDER"]="PCI_BUS_ID"
os.environ["CUDA_VISIBLE_DEVICES"]="0"

import napari
import trimesh
import pyvista as pv
import pyacvd

In [ ]:
fix_mesh = trimesh.load("/home/tmurakami/src/surface_mesh/human_tissue_morphpaper/220715_prefrontal_q2_R01_pia.ply")
mov_mesh = trimesh.load("/home/tmurakami/src/surface_mesh/human_tissue_morphpaper/220715_prefrontal_q2_R01_wm.ply")

### Fix the z resolution problem
fix_mesh.vertices = fix_mesh.vertices * np.asarray([1.5,1.0,1.0])
mov_mesh.vertices = mov_mesh.vertices * np.asarray([1.5,1.0,1.0])
target_area = 10000 # 100 micron x 100 micron
n_target_mov = int((mov_mesh.area // target_area) // 2) # division by two to convert number of faces to number of verts
n_target_fix = int((fix_mesh.area // target_area) // 2) # division by two to convert number of faces to number of verts

In [ ]:
# --- trimesh -> pyvista ---
V, F = mov_mesh.vertices, mov_mesh.faces
faces_pv = np.hstack([np.full((len(F), 1), 3, dtype=np.int64), F]).ravel()
pmesh = pv.PolyData(V, faces_pv)

# --- uniform remeshing ---
clus = pyacvd.Clustering(pmesh)
clus.subdivide(3)                 # densify first so clustering has points to work with
clus.cluster(n_target_mov)
remesh = clus.create_mesh()

# --- pyvista -> trimesh ---
faces_tm = remesh.faces.reshape(-1, 4)[:, 1:]   # drop the leading "3" per face
mov_mesh = trimesh.Trimesh(remesh.points, faces_tm, process=False)


# --- trimesh -> pyvista ---
V, F = fix_mesh.vertices, fix_mesh.faces
faces_pv = np.hstack([np.full((len(F), 1), 3, dtype=np.int64), F]).ravel()
pmesh = pv.PolyData(V, faces_pv)

# --- uniform remeshing ---
clus = pyacvd.Clustering(pmesh)
clus.subdivide(3)                 # densify first so clustering has points to work with
clus.cluster(n_target_fix)
remesh = clus.create_mesh()

# --- pyvista -> trimesh ---
faces_tm = remesh.faces.reshape(-1, 4)[:, 1:]   # drop the leading "3" per face
fix_mesh = trimesh.Trimesh(remesh.points, faces_tm, process=False)

In [ ]:
fix_vertices = fix_mesh.vertices
mov_vertices = mov_mesh.vertices
fix_face = fix_mesh.faces
mov_face = mov_mesh.faces

fix_vertex_colors = np.tile(
    np.array([0.0, 1.0, 0.0, 1.0]),  # RGBA: red
    (fix_vertices.shape[0], 1)
)
mov_vertex_colors = np.tile(
    np.array([1.0, 0.0, 1.0, 1.0]),  # RGBA: red
    (mov_vertices.shape[0], 1)
)

In [ ]:
viewer = napari.Viewer(ndisplay=3)
viewer.add_points(fix_vertices, size=50, face_color='lime', blending="additive")
viewer.add_points(mov_vertices, size=50, face_color='magenta', blending="additive")
viewer.add_surface((fix_vertices,fix_face),blending="additive",vertex_colors=fix_vertex_colors,shading='smooth')
viewer.add_surface((mov_vertices,mov_face),blending="additive",vertex_colors=mov_vertex_colors,shading='smooth')

In [ ]:
mov_mesh.export('/home/tmurakami/src/flow_analysis/human_analysis/01_output/220715_prefrontal_q2_R01_wm_refined.ply')
fix_mesh.export('/home/tmurakami/src/flow_analysis/human_analysis/01_output/220715_prefrontal_q2_R01_pia_refined.ply')